# Who Keeps Texas On After Dark?

Day 17 · Light & dark. One sunny day on ERCOT, Texas's main grid, hour by hour. The land darkens with the real sun,
and every power plant glows with how hard its fuel is working across the grid at that moment.
Run `python scripts\fetch_week3.py day13 day17` first.

## 1 · Settings

In [ ]:
DAY      = 17
PROMPT   = "Light & dark"
SLUG     = "light-dark-texas-grid"
NOTEBOOK = "day17_light_dark_texas_grid.ipynb"
TITLE    = "Who Keeps Texas\nOn After Dark?"
CREDITS  = ("Hourly generation by fuel: U.S. Energy Information Administration, Form EIA-930 (ERCOT). "
            "Power plants: EIA-860M (September 2026). Texas boundary: US Census Bureau. Sun position computed from date and time.")

THE_DAY  = None          # None = the sunniest day in the file (most solar MWh). Or a date like "07/15/2026"
FPS      = 18
PER_HOUR = 5             # frames for each hour of the day
HOLD     = 1.5           # seconds to pause on the last frame

PAGE, INK, SOFT = "#0c1016", "#f1ede4", "#8b949c"
NIGHT, DUSK, DAYLIGHT = "#151d2b", "#3b3552", "#c9d3d6"      # land colour as the sun moves
FUELS = {   # fuel: (colour, label). Plants of other fuels sit as faint dots.
    "Gas":     ("#ff7a3d", "Gas"),
    "Solar":   ("#ffd34d", "Solar"),
    "Wind":    ("#6ee7d2", "Wind"),
    "Storage": ("#b38cff", "Batteries"),
}
OTHER = {"Coal": "#a1887f", "Nuclear": "#e7e7e7", "Hydro": "#4fa3e0", "Oil": "#c0573e", "Other": "#7b858c"}

## 2 · Load

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.path import Path as MPath
from matplotlib.patches import PathPatch
from mapkit import ai, poster, publish
from mapkit.paths import PROCESSED

hourly = pd.read_csv(PROCESSED / "ercot_hourly_fuel.csv", parse_dates=["utc_end"])
gens = pd.read_csv(PROCESSED / "eia860m_generators.csv")
texas = ai.load_states().query("state_abb == 'TX'").to_crs(4326)

gens = gens[(gens.ba == "ERCO") & (gens.status_group == "operating") & ~gens.status.str.startswith("(OS)")]
plants = gens.groupby(["plant_id", "fuel"]).agg(mw=("mw", "sum"), lat=("lat", "first"), lon=("lon", "first")).reset_index()
CAPACITY = plants.groupby("fuel").mw.sum()                 # all of ERCOT, to match the grid totals
shown = plants[plants.plant_id.isin(gens[gens.state == "TX"].plant_id)]   # draw the Texas ones (a few sit in Oklahoma)
print(f"ERCOT: {len(plants):,} plant-fuel pairs, {CAPACITY.sum()/1000:,.0f} GW")
print((CAPACITY / 1000).round(1).sort_values(ascending=False).to_string())

## 3 · Analysis, every number on the map comes from here

In [ ]:
fuel_cols = [c for c in hourly.columns if c not in ("utc_end", "local_date", "hour", "demand")]
daily_solar = hourly.groupby("local_date").Solar.sum()
day = THE_DAY or daily_solar.idxmax()
d = hourly[hourly.local_date == day].sort_values("utc_end").reset_index(drop=True)
assert len(d) >= 23, f"{day} has only {len(d)} hours"
d["mid"] = d.utc_end - pd.Timedelta(minutes=30)                       # middle of each hour
d["local"] = d.mid.dt.tz_localize("UTC").dt.tz_convert("America/Chicago")
total = d[fuel_cols].clip(lower=0).sum(axis=1)
share = d[fuel_cols].clip(lower=0).div(total, axis=0) * 100

def clock(ts, minutes=True):
    """7:40 PM style, works on Windows too."""
    h = ts.hour % 12 or 12
    return f"{h}:{ts.minute:02d} {'AM' if ts.hour < 12 else 'PM'}" if minutes else f"{h} {'AM' if ts.hour < 12 else 'PM'}"

noon = share.Solar.idxmax()
evening = d.index[d.local.dt.hour >= 17]
gas_peak = share.loc[evening, "Gas"].idxmax()
batt_peak = d.Storage.idxmax() if "Storage" in d else None
STATS = dict(
    date=f"{d.local.iloc[12]:%B} {d.local.iloc[12].day}, {d.local.iloc[12].year}",
    solar_peak_share=share.Solar.max(), solar_peak_time=d.local[noon],
    gas_at_solar_peak=share.Gas[noon],
    gas_evening_share=share.Gas[gas_peak], gas_evening_time=d.local[gas_peak], solar_at_gas_peak=share.Solar[gas_peak],
    batt_peak_mw=d.Storage.max() if "Storage" in d else 0, batt_peak_time=d.local[batt_peak] if batt_peak is not None else None,
)
print(f"Day: {day}")
print(f"Solar peaks at {STATS['solar_peak_share']:.0f}% of generation at {clock(STATS['solar_peak_time'], False)}; gas is {STATS['gas_at_solar_peak']:.0f}% then")
print(f"After dark gas peaks at {STATS['gas_evening_share']:.0f}% at {clock(STATS['gas_evening_time'], False)}; solar is {STATS['solar_at_gas_peak']:.0f}% then")
if batt_peak is not None:
    print(f"Batteries discharge most at {clock(STATS['batt_peak_time'], False)}: {STATS['batt_peak_mw']:,.0f} MW")

## 4 · Map, one frame at a time

In [ ]:
F = poster.font
TITLE_F, TEXT_F, TEXT_B = poster.BEBAS, poster.SANS, poster.SANS_BOLD
land_cmap = LinearSegmentedColormap.from_list("sun", [(0, NIGHT), (0.45, NIGHT), (0.6, DUSK), (0.78, DAYLIGHT), (1, DAYLIGHT)])

def sun_elevation(lat, lon, when_utc):
    """Sun height above the horizon in degrees (NOAA's simple formula, good to about a degree)."""
    n = when_utc.timestamp() / 86400 + 2440587.5 - 2451545.0
    L = np.radians((280.460 + 0.9856474 * n) % 360)
    g = np.radians((357.528 + 0.9856003 * n) % 360)
    lam = L + np.radians(1.915) * np.sin(g) + np.radians(0.020) * np.sin(2 * g)
    eps = np.radians(23.439 - 0.0000004 * n)
    dec = np.arcsin(np.sin(eps) * np.sin(lam))
    ra = np.arctan2(np.cos(eps) * np.sin(lam), np.cos(lam))
    gmst = (18.697374558 + 24.06570982441908 * n) % 24
    ha = np.radians(gmst * 15 + lon) - ra
    la = np.radians(lat)
    return np.degrees(np.arcsin(np.sin(la) * np.sin(dec) + np.cos(la) * np.cos(dec) * np.cos(ha)))

w, s, e, n_ = texas.total_bounds
GLON, GLAT = np.meshgrid(np.linspace(w, e, 220), np.linspace(n_, s, 180))
ASPECT = 1 / np.cos(np.radians((s + n_) / 2))
tx_path = MPath.make_compound_path(*[MPath(np.asarray(p.exterior.coords))
                                     for p in getattr(texas.geometry.iloc[0], "geoms", [texas.geometry.iloc[0]])])

def at(t):
    """Values at fractional hour index t (0 = first hour's middle), by linear interpolation."""
    i = int(np.clip(np.floor(t), 0, len(d) - 2)); f = t - i
    row = d.loc[i, fuel_cols] * (1 - f) + d.loc[i + 1, fuel_cols] * f
    when = d.mid[i] + (d.mid[i + 1] - d.mid[i]) * f
    return row, when

def draw(t):
    row, when = at(t)
    fig = plt.figure(figsize=(10.8, 13.5), dpi=100, facecolor=PAGE)
    ax = fig.add_axes([0.04, 0.275, 0.92, 0.50], facecolor=PAGE)
    elev = sun_elevation(GLAT, GLON, pd.Timestamp(when))          # times are UTC
    img = ax.imshow(np.clip((elev + 12) / 22, 0, 1), cmap=land_cmap, vmin=0, vmax=1, extent=(w, e, s, n_), zorder=1,
                    interpolation="bilinear")
    img.set_clip_path(PathPatch(tx_path, transform=ax.transData, fc="none", ec="none"))
    texas.boundary.plot(ax=ax, color=SOFT, lw=0.6, zorder=2)
    for fuel, col in OTHER.items():
        p = shown[shown.fuel == fuel]
        ax.scatter(p.lon, p.lat, s=np.sqrt(p.mw) * 1.2, color=col, alpha=0.35, lw=0, zorder=3)
    for fuel, (col, _) in FUELS.items():
        if fuel not in row:
            continue
        p = shown[shown.fuel == fuel]
        k = float(np.clip(row[fuel] / CAPACITY.get(fuel, np.inf), 0, 1))       # how hard this fuel is working, 0 to 1
        base = np.sqrt(p.mw)
        ax.scatter(p.lon, p.lat, s=base * 1.2, color=col, alpha=0.18, lw=0, zorder=4)               # always visible, faint
        ax.scatter(p.lon, p.lat, s=base * (1.5 + 9 * k), color=col, alpha=0.10 + 0.25 * k, lw=0, zorder=5)   # glow
        ax.scatter(p.lon, p.lat, s=base * (0.8 + 2.2 * k), color=col, alpha=min(1, 0.25 + k), lw=0, zorder=6)
    ax.set_xlim(w - 0.2, e + 0.2); ax.set_ylim(s - 0.2, n_ + 0.2); ax.set_aspect(ASPECT); ax.set_axis_off()

    # title and clock
    fig.text(0.06, 0.965, TITLE.upper(), color=INK, fontproperties=F(TITLE_F, 68), va="top", linespacing=0.92)
    local = pd.Timestamp(when).tz_localize("UTC").tz_convert("America/Chicago")
    fig.text(0.94, 0.965, clock(local), color=INK, fontproperties=F(TITLE_F, 64), ha="right", va="top")
    fig.text(0.94, 0.905, f"{STATS['date']} · Central Time", color=SOFT, fontproperties=F(TEXT_F, 13), ha="right", va="top")

    # legend: one row, with this moment's share of generation
    tot = row.clip(lower=0).sum()
    for i, (fuel, (col, lab)) in enumerate(FUELS.items()):
        x = 0.06 + i * 0.18
        val = row.get(fuel, 0)
        fig.text(x, 0.805, lab, color=col, fontproperties=F(TEXT_B, 14), va="top")
        fig.text(x, 0.782, f"{max(val, 0) / tot * 100:.0f}% of power", color=SOFT, fontproperties=F(TEXT_F, 12.5), va="top")

    base = sum(max(row.get(f, 0), 0) for f in ("Coal", "Nuclear"))
    fig.text(0.06 + 4 * 0.18, 0.805, "Coal & nuclear", color=OTHER["Coal"], fontproperties=F(TEXT_B, 14), va="top")
    fig.text(0.06 + 4 * 0.18, 0.782, f"{base / tot * 100:.0f}% of power", color=SOFT, fontproperties=F(TEXT_F, 12.5), va="top")

    # the day as a stacked chart, with a line for now
    cx = fig.add_axes([0.06, 0.09, 0.88, 0.15], facecolor=PAGE)
    order = [f for f in ["Nuclear", "Coal", "Hydro", "Other", "Oil", "Wind", "Gas", "Solar", "Storage"] if f in d]
    cols = [FUELS.get(f, (OTHER.get(f, SOFT),))[0] for f in order]
    hrs = np.arange(len(d))
    cx.stackplot(hrs, *[d[f].clip(lower=0) / 1000 for f in order], colors=cols, alpha=0.9, lw=0)
    cx.axvline(t, color=INK, lw=1.4)
    cx.set_xlim(0, len(d) - 1); cx.set_ylim(0, None)
    cx.set_xticks([i for i in hrs if d.local[i].hour % 6 == 0])
    cx.set_xticklabels([clock(d.local[i], False) for i in hrs if d.local[i].hour % 6 == 0], color=SOFT,
                       fontproperties=F(TEXT_F, 11))
    cx.tick_params(axis="y", colors=SOFT, labelsize=10, length=0); cx.tick_params(axis="x", length=0)
    for sp in cx.spines.values():
        sp.set_visible(False)
    cx.set_ylabel("GW", color=SOFT, fontproperties=F(TEXT_F, 11))
    for lab in cx.get_yticklabels():
        lab.set_fontproperties(F(TEXT_F, 10))

    fig.text(0.06, 0.262, "Land shading follows the real sun. Each plant glows with how hard its fuel is working across the grid "
             "that hour, not its own meter.", color=SOFT, fontproperties=F(TEXT_F, 11), va="top", wrap=True)
    poster.credit(fig, "EIA-930 hourly generation (ERCOT) · EIA-860M plants · US Census", x=0.06, y=0.018, color=SOFT,
                  path=TEXT_F, name_path=TEXT_B, name_color=INK, size=9.5)
    return fig

fig = draw(float(gas_peak))
plt.show()

## 5 · Export: MP4 for LinkedIn, plus a still at the evening gas peak

In [ ]:
import io, imageio.v2 as imageio
from PIL import Image
from mapkit.paths import OUTPUT

ts = np.arange(0, len(d) - 1 + 1e-9, 1 / PER_HOUR)
ts = np.r_[ts, [ts[-1]] * int(HOLD * FPS)]
mp4 = OUTPUT / f"day{DAY:02d}_{SLUG}.mp4"
with imageio.get_writer(mp4, fps=FPS, codec="libx264", quality=None, pixelformat="yuv420p",
                        ffmpeg_params=["-crf", "18", "-preset", "veryfast"]) as out:
    for k, t in enumerate(ts):
        f = draw(t)
        buf = io.BytesIO(); f.savefig(buf, format="png", facecolor=PAGE); plt.close(f)
        out.append_data(np.array(Image.open(buf).convert("RGB")))
print("saved", mp4)

still = draw(float(gas_peak))
publish.save(still, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle=f"Solar peaks at {STATS['solar_peak_share']:.0f}% at midday; after dark gas carries {STATS['gas_evening_share']:.0f}%",
             series="ai", tool="Python", notebook=NOTEBOOK, credits=CREDITS)